# VisionXM ResNet18 GPU Training
Uses ImageFolder on the provided folder structure.


In [ ]:
import torch
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
from pathlib import Path
import torch.nn as nn
from tqdm import tqdm
from PIL import Image


In [23]:
DATA=Path('data')
train_dir=DATA/'train'

In [24]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

Using: cuda


In [25]:
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485,0.456,0.406],
        [0.229,0.224,0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485,0.456,0.406],
        [0.229,0.224,0.225]
    )
])

In [40]:
train_samples = []
test_samples = []

# ---------- TRAIN (GOOD ONLY) ----------
good_train = Path("data/train/good")

for img in good_train.glob("*"):
    if img.is_file():
        train_samples.append((str(img), 0))

# ---------- TEST (GOOD + DEFECTIVE) ----------
test_root = Path("data/test")

for folder in test_root.iterdir():

    if not folder.is_dir():
        continue

    label = 0 if folder.name == "good" else 1

    for img in folder.glob("*"):
        if img.is_file():
            test_samples.append((str(img), label))

print("Training images:", len(train_samples))
print("Testing images :", len(test_samples))

Training images: 320
Testing images : 160


In [42]:
class ListDataset(Dataset):

    def __init__(self, samples, transform):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        img_path, label = self.samples[idx]

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label


train_dataset = ListDataset(train_samples, train_transform)
test_dataset = ListDataset(test_samples, test_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

In [43]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

model.fc = nn.Linear(model.fc.in_features, 2)

model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [44]:
epochs = 15

best_acc = 0

for epoch in range(epochs):

    model.train()

    running_loss = 0

    for images, labels in tqdm(train_loader):

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    ##########################

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            preds = outputs.argmax(1)

            correct += (preds == labels).sum().item()

            total += labels.size(0)

    accuracy = 100 * correct / total

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Loss={running_loss/len(train_loader):.4f} | "
        f"Val Accuracy={accuracy:.2f}%"
    )

    if accuracy > best_acc:

        best_acc = accuracy

        torch.save(
            model.state_dict(),
            "best_resnet18_binary.pth"
        )

100%|██████████| 10/10 [00:06<00:00,  1.61it/s]


Epoch 1/15 | Loss=0.4517 | Val Accuracy=45.31%


100%|██████████| 10/10 [00:12<00:00,  1.25s/it]


Epoch 2/15 | Loss=0.0995 | Val Accuracy=100.00%


100%|██████████| 10/10 [00:14<00:00,  1.41s/it]


...

Epoch 13/15 | Loss=0.0029 | Val Accuracy=100.00%


100%|██████████| 10/10 [00:14<00:00,  1.40s/it]


Epoch 14/15 | Loss=0.0028 | Val Accuracy=100.00%


100%|██████████| 10/10 [00:13<00:00,  1.35s/it]


Epoch 15/15 | Loss=0.0024 | Val Accuracy=100.00%


In [45]:
# Path to image
image_path = "D:\\Projects\\VisionXM\\Defected screw.png"      # <-- change this

# Load image
image = Image.open(image_path).convert("RGB")

# Apply same test transform
input_tensor = test_transform(image).unsqueeze(0).to(device)

# Predict
model.eval()
with torch.no_grad():
    output = model(input_tensor)
    probs = torch.softmax(output, dim=1)
    pred = torch.argmax(probs, dim=1).item()

classes = {0: "Good", 1: "Defective"}

print(f"Prediction : {classes[pred]}")
print(f"Confidence : {probs[0][pred].item()*100:.2f}%")
print(f"Probabilities -> Good: {probs[0][0]:.4f}, Defective: {probs[0][1]:.4f}")

Prediction : Good
Confidence : 99.61%
Probabilities -> Good: 0.9961, Defective: 0.0039


In [46]:
# Path to image
image_path = "D:\\Projects\\VisionXM\\Good screw.png"      # <-- change this

# Load image
image = Image.open(image_path).convert("RGB")

# Apply same test transform
input_tensor = test_transform(image).unsqueeze(0).to(device)

# Predict
model.eval()
with torch.no_grad():
    output = model(input_tensor)
    probs = torch.softmax(output, dim=1)
    pred = torch.argmax(probs, dim=1).item()

classes = {0: "Good", 1: "Defective"}

print(f"Prediction : {classes[pred]}")
print(f"Confidence : {probs[0][pred].item()*100:.2f}%")
print(f"Probabilities -> Good: {probs[0][0]:.4f}, Defective: {probs[0][1]:.4f}")

Prediction : Good
Confidence : 99.54%
Probabilities -> Good: 0.9954, Defective: 0.0046
